# Deep Researcher Agent

This notebook focuses on the Deep Research Agent which has been built using [LangChain Deep Agents harness](https://docs.langchain.com/oss/python/deepagents/harness). It will demonstrate how to get started with the NVIDIA AI-Q blueprint - deep researcher agent workflow: with **web search only**. You will:

1. **Understand** the deep researcher architecture (orchestrator, planner, researcher).  
2. **Run** the deep research agent using the Python API
3. **Run** the deep research agent via the NeMo Agent Toolkit CLI (`nat run`) for interactive research with a minimal config.
4. Run **`nat eval`** on a benchmark dataset and interpret results.
5. Use **profiling/tracing** (such as Phoenix/Weave) if configured.

# Table of Contents

- [About the deep researcher](#about)  
- [Prerequisites](#prereqs)  
- [Simplest way to run the agent (Python API)](#simplest-run)  
- [Simplified config (web search only)](#config)  
- [Run with NAT CLI](#nat-run)  
- [Evaluate with NAT Eval](#nat-eval)
- [Profiling and tracing](#profiling)  
- [Next steps](#next)  


<a id="about"></a>
## About the deep researcher

The deep researcher is implemented in `src/aiq_agent/agents/deep_researcher/agent.py`. It uses the [LangChain DeepAgents](https://docs.langchain.com/oss/python/deepagents/overview) library. The architecture consists of one main agent and two subagents. The system makes use of `TodoListMiddleware`, `FilesystemMiddleware` and so on for context isolation and management during the execution of a long-running agent.

- **Orchestrator** – Coordinates the workflow and produces the final report.  
- **Planner subagent** – Builds a structured research plan and search queries.  
- **Researcher subagent** – Runs search queries and pulls in content from tools (e.g. web search).
Read more about it in this [documentation](../source/architecture/agents/deep-researcher.md#workflow-phases)

Phases: 
- (1) planning
- (2) research loops
- (3) final report.

With **web search only**, the researcher uses a single tool (e.g. Tavily) to gather information.

<a id="prereqs"></a>
## Prerequisites

- **Python 3.11–3.13**
- **[uv](https://github.com/astral-sh/uv)** package manager
- **LLM API key** – For your chosen provider (required for LLM inference):
    - **NVIDIA API Key** from [NVIDIA AI](https://build.nvidia.com/) (for NIM models)
Optional requirements:
- **Web search API key**:
  - **Tavily** – [tavily.com](https://tavily.com) (for general web search)
- **Node.js 18+ and npm** (optional, for web UI mode)

1. **Clone** the NVIDIA AI-Q Blueprint repository
>**Note**: Skip this step if you are not on brev and instead running from within the repository.

In [ ]:
%%bash
git clone https://github.com/NVIDIA-AI-Blueprints/aiq.git && cd aiq

2. **Install** dependencies
>**Note**: Skip this step if you already have a virtual environment setup. 

>Just run:
>```bash
>source .venv/bin/activate
>```

In [ ]:
# Run setup from the project root (where pyproject.toml and scripts/setup.sh live).
import os
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "scripts" / "setup.sh").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing pyproject.toml and scripts/setup.sh")

os.environ["REPO_ROOT"] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
os.environ["UV_VENV_CLEAR"] = "1"
get_ipython().system("./scripts/setup.sh")

In [ ]:
%%bash
source .venv/bin/activate

4. **Obtain and set** API keys


| API | Environment variable | Purpose |
|-----|----------------------|--------|
| **NVIDIA** | `NVIDIA_API_KEY` | NIM / NGC model inference (required) |
| **Tavily** | `TAVILY_API_KEY` | Web search (recommended) |

**NVIDIA API key:**  
1. Go to [NVIDIA Build](https://build.nvidia.com/explore/discover) or [NGC](https://ngc.nvidia.com/).  
2. Sign in and open a model card (e.g. Nemotron).  
3. Use **Get API Key** → **Generate Key**, then copy the key (starts with `nvapi-`).  

**Tavily API key:**  
1. Go to [tavily.com](https://tavily.com) and create an account.  
2. Create an API key in the dashboard and add it to your environment. 

In [ ]:
import os
from pathlib import Path

if os.environ.get("REPO_ROOT"):
    os.chdir(os.environ["REPO_ROOT"])
else:
    cwd = Path.cwd()
    if not (cwd / "configs").exists() and not (cwd / "src").exists():
        root = cwd.parent.parent
        if (root / "configs").is_dir() or (root / "src").is_dir():
            os.chdir(root)
print("Working directory:", os.getcwd())

In [ ]:
import getpass

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

if "TAVILY_API_KEY" not in os.environ:
    tavily_api_key = getpass.getpass("Enter your Tavily API key: ")
    os.environ["TAVILY_API_KEY"] = tavily_api_key

<a id="simplest-run"></a>
## Simplest way to run the agent (Python API)

You can run the deep researcher agent directly in Python without a config file: create an LLM, a web search tool, wire them with `LLMProvider`, and call `agent.run(state)`. The next section walks through what each part does and what happens under the hood.

### Walkthrough: what the code does

- **LLM and provider:** You create one chat model (e.g. `ChatNVIDIA`) and a single `LLMProvider`. The provider is the central place that assigns which LLM runs which role. You call `provider.set_default(llm)` and then `provider.configure(LLMRole.ORCHESTRATOR, llm)` (and similarly for `PLANNER`, `RESEARCHER`). That way the orchestrator, planner, and researcher can all share the same model, or you can later point each role at a different model.

- **Tools:** You pass a list of LangChain tools (here, only `TavilySearch`). The agent adds an internal `think` tool and passes the full list to both the orchestrator and the subagents. The orchestrator uses tools to track progress (`write_todos`), delegate work (`task`), and read/write files; the researcher uses the same list to run search (e.g. `tavily_search`) and `think`.

- **Agent and state:** `DeepResearcherAgent(llm_provider=..., tools=[...])` builds the middleware and loads prompts; it does *not* build the LangGraph yet. `DeepResearchAgentState(messages=[HumanMessage(...)])` is the input state. When you call `agent.run(state)`, the agent builds the orchestrator graph (see below), invokes it, checks report completeness, and may retry with feedback up to 5 times.

### What happens internally when you call `agent.run(state)`

1. **Build the orchestrator:** `run()` calls `_build_orchestrator_agent(state)`. That method loads the orchestrator prompt (Jinja2) with the current tools and state (e.g. `available_documents`), defines a backend for the deep agent (in-memory state plus a `/shared/` route for plan and scratch files), and calls the [deepagents](https://docs.langchain.com/oss/python/deepagents/overview) library’s `create_deep_agent()` with the orchestrator LLM, tools, **subagents** (see snippet below), **middleware** (see snippet below), and store. So the “agent” you run is a LangGraph compiled from that deep agent spec.

2. **Invoke the graph:** The code runs `await agent.ainvoke(state, config={...})`. The orchestrator node runs first; it may call `write_todos`, then `task(description=..., subagent_type="planner-agent")`. That triggers the planner subagent, which has its own LLM call and tools (same tool list). The planner typically uses search tools and `write_file` to produce `/shared/plan.json`. Control returns to the orchestrator, which may then call `task(..., subagent_type="researcher-agent")` multiple times for different queries. Each researcher run uses the researcher LLM and the same tools to run search and synthesize. The orchestrator reads plan and research outputs, runs more tools (`think`, `read_file`, `grep`), and eventually produces the final report in its last message.

3. **Report completeness and retries:** After each `ainvoke`, the code calls `_is_report_complete(result)`. The report is considered complete only if the last message has at least 1500 characters, at least two `##` section headers, and a Sources/References section, and does not contain phrases like “please confirm” or “do you want me to”. If incomplete, the code appends a `HumanMessage` with feedback (e.g. “include a Sources section”) and calls `ainvoke` again, up to 5 times.

4. **Return:** The final state (with all messages, including the final report) is returned as a `DeepResearchAgentState`; the last AI message content is the report.

### Subagent definitions (from `agent.py`)

The orchestrator delegates to two subagents. They are built in `_get_subagents(state)` and passed to `create_deep_agent()`. Each subagent has its own **name**, **description** (used when the orchestrator chooses a subagent), **system_prompt** (Jinja2-rendered with `tools`, `available_documents`, etc.), **tools** (the same list: `think` + your search tools), **model** (from the provider for that role), and **middleware** (same stack as the orchestrator). Conceptually:

```python
# _get_subagents(state) returns a list of two dicts:
[
    {
        "name": "planner-agent",
        "description": "Content-driven research planning - iteratively builds evidence-grounded "
                       "outlines through interleaved search and outline optimization",
        "system_prompt": render_prompt_template(self._prompts["planner"], tools=..., available_documents=...),
        "tools": self.all_tools,   # think + your tools (e.g. tavily_search)
        "model": self.llm_provider.get(LLMRole.PLANNER),
        "middleware": self.middleware,
    },
    {
        "name": "researcher-agent",
        "description": "Information gathering - executes search queries and synthesizes "
                       "relevant content from available sources",
        "system_prompt": render_prompt_template(self._prompts["researcher"], current_datetime=..., tools=..., available_documents=...),
        "tools": self.all_tools,
        "model": self.llm_provider.get(LLMRole.RESEARCHER),
        "middleware": self.middleware,
    },
]
```

The **planner** prompt (from `prompts/planner.j2`) tells the model to analyze the task, use search tools to discover what exists, and write a structured plan to `/shared/plan.json` (task analysis, TOC, constraints, queries). The **researcher** prompt (from `prompts/researcher.j2`) tells the model to run search for a given query, synthesize findings, and cite sources. The orchestrator’s prompt (from `prompts/orchestrator.j2`) describes the workflow steps (write_todos → planning → research → verify → synthesize → write report) and when to call each subagent.

### Middleware (from `agent.py`)

All three agents (orchestrator, planner, researcher) use the same middleware list. It runs in order around model calls:

```python
self.middleware = [
    EmptyContentFixMiddleware(),
    ToolNameSanitizationMiddleware(valid_tool_names=[t.name for t in self.all_tools]),
    ModelRetryMiddleware(max_retries=10, backoff_factor=2.0, initial_delay=1.0),
]
```

- **EmptyContentFixMiddleware:** Some LLM APIs reject messages with empty content. This middleware runs before the request is sent to the model: it finds any `ToolMessage` with empty `content` and replaces it with a placeholder string (e.g. `"empty content received."`) so the API call does not fail.

- **ToolNameSanitizationMiddleware:** LLMs sometimes emit malformed or hallucinated tool names (e.g. with `<|channel|>` suffixes or names like `open_file` instead of `read_file`). This middleware runs after the model response: it rewrites tool names in `AIMessage.tool_calls` to match the allowed `valid_tool_names` (and a small alias map), so the runtime dispatches the correct tool.

- **ModelRetryMiddleware:** LangChain’s built-in retry for model calls: on failure it retries up to 10 times with exponential backoff (initial_delay 1s, backoff_factor 2.0), so transient API errors are retried automatically.

In [ ]:
# Simplest run: one LLM, one web search tool, then agent.run(state)
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_tavily import TavilySearch

from aiq_agent.agents.deep_researcher.agent import DeepResearcherAgent
from aiq_agent.agents.deep_researcher.models import DeepResearchAgentState
from aiq_agent.common import LLMProvider
from aiq_agent.common import LLMRole
from aiq_agent.common import VerboseTraceCallback

# One LLM for all roles (orchestrator, planner, researcher)
llm = ChatNVIDIA(
    model="nvidia/nemotron-3-nano-30b-a3b",
    base_url="https://integrate.api.nvidia.com/v1",
    temperature=1.0,
    top_p=1.0,
    max_completion_tokens=256000,
)
provider = LLMProvider()
provider.set_default(llm)
provider.configure(LLMRole.ORCHESTRATOR, llm)
provider.configure(LLMRole.RESEARCHER, llm)
provider.configure(LLMRole.PLANNER, llm)

# Web search tool (uses TAVILY_API_KEY from env)
web_search_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
)


@tool("advanced_web_search_tool")
def tavily_search(query: str) -> dict:
    """Search the web with Tavily using a plain text query."""
    # Keep the tool signature strict to avoid malformed optional args.
    return web_search_tool.invoke({"query": query})


agent = DeepResearcherAgent(
    llm_provider=provider,
    tools=[tavily_search],
    max_loops=2,
    verbose=True,
    callbacks=[VerboseTraceCallback()],
)
state = DeepResearchAgentState(
    messages=[
        HumanMessage(
            content="Conduct a comprehensive technical comparison between NVIDIA GB300 and Google Ironwood AI accelerator platforms. Generate a structured report with the following sections: Compute Architecture, Memory Subsystem, Interconnect & Scalability, System & Power, Software Ecosystem and Benchmarks"
        )
    ]
)

result = await agent.run(state)

# Final report is in the last AI message
from langchain_core.messages import AIMessage

last_ai = next((m for m in reversed(result.messages) if isinstance(m, AIMessage) and m.content), None)
if last_ai:
    print(last_ai.content[:2000] + ("..." if len(last_ai.content) > 2000 else ""))

<a id="config"></a>
## Using NeMo Agent Toolkit to wrap the Agent


**Why use the NeMo Agent toolkit (NAT) for agent development and optimization?** NAT gives you a single YAML-based configuration for LLMs, tools, and workflows so you can swap models and tools without code changes. The **`nat eval`** command runs your workflow on a dataset and scores it with built-in or custom evaluators, and the toolkit’s [evaluation docs](https://docs.nvidia.com/nemo/agent-toolkit/latest/improve-workflows/evaluate.html) describe how to add evaluators and benchmarks. For optimization, NAT’s pipeline (including **`nat optimize`**) supports tuning prompts and hyperparameters against benchmarks through integrations such as genetic-algorithm prompt search and numeric trial search. Profiling and tracing are built in: enable the profiler in your eval config to get latency reports, token metrics, and traces that you can send to observability backends (Phoenix, Weave, or others). For full details, see the [NeMo Agent toolkit documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/).

## Simplified config (web search only)

Here, we will create a simplified configuration that uses one LLM and one tool that focuses only on web search. The web search tool uses **Tavily web search** for internet search capabilities. No paper search, no knowledge layer.

In [ ]:
%%writefile config_simple_deep_researcher.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

llms:
  nemotron_llm_deep:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 1.0
    top_p: 1.0 
    max_tokens: 128000
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

functions:
  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 5
    advanced_search: true

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_llm_deep
    max_loops: 2
    tools:
      - advanced_web_search_tool

workflow:
  _type: deep_research_workflow

<a id="nat-run"></a>
## Run with NAT CLI

**Basic Execution**

In [ ]:
!.venv/bin/nat run \
    --config_file config_simple_deep_researcher.yml \
    --input "Conduct a comprehensive technical comparison between NVIDIA GB300 and Google Ironwood AI accelerator platforms. Generate a structured report with the following sections: Compute Architecture, Memory Subsystem, Interconnect & Scalability, System & Power, Software Ecosystem and Benchmarks"

<div class="alert alert-block alert-info">
<b>Note:</b> Deep research can take several minutes: multiple LLM calls and web searches run in sequence. Watch the logs to see planning and research steps.
</div>

## Evaluate the Workflow with Nat eval

The `nat eval` command runs the workflow against a dataset and evaluates the results using specified metrics. This is essential for measuring research quality and factual accuracy. The AgentIQ repository includes a DeepResearch Bench integration that uses RACE (report quality) and FACT (citation accuracy) metrics.

### Obtain the dataset files:

1. Download from the [DeepResearch Bench GitHub repository](https://github.com/Ayanami0730/deep_research_bench)
2. Create a directory called `data` under `frontends/benchmarks/deepresearch_bench/`
2. Place the files in `frontends/benchmarks/deepresearch_bench/data/`:
   - `drb_full_dataset.json` (required)
   - `criteria.jsonl` (required)

In [ ]:
%load frontends/benchmarks/deepresearch_bench/configs/config_deep_research_bench.yml

1. **Choose a judge model** – Use a capable model for consistent scoring, e.g.:
   - **OpenAI** – GPT-4o, GPT-5 (via `OPENAI_API_KEY`)
   - **Gemini** – Gemini 2.5 Pro or Flash

2. **Obtain an API key** for the provider you chose (OpenAI or Gemini).

3. **Set the key**:
   ```bash
   # For OpenAI judge 
   OPENAI_API_KEY=your_openai_key

   # For Gemini judge 
   GEMINI_API_KEY=your_gemini_key
   ```

4. **Use a different judge in the config** – Update `llms:` in the config and set `eval.evaluators.race.llm_name` to that LLM name. Ensure the corresponding API key is set.


In [ ]:
%%writefile docs/notebooks/config_deep_research_bench.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO
  use_uvloop: true

llms:
  nemotron_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 1.0
    top_p: 1.0
    max_tokens: 128000
    max_retries: 10
    timeout: 600
    chat_template_kwargs:
      enable_thinking: true

  # Update this section with the name of the judge model you chose
  judge_llm:
    _type: openai
    model_name: "gpt-5.2" 
    api_key: "<replace with your API key or read from environment variable>"
    timeout: 600
    max_retries: 10
    temperature: 0.1

functions:
  paper_search_tool:
    _type: paper_search
    max_results: 5
    serper_api_key: ${SERPER_API_KEY}

  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 2
    advanced_search: true

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_llm
    max_loops: 2
    tools:
      - paper_search_tool
      - advanced_web_search_tool

workflow:
  _type: deep_research_workflow

eval:
  general:
    workflow_alias: "aiq-deepresearcher"
    output_dir: notebooks/results
    max_concurrency: 4
    dataset:
      _type: json
      file_path: frontends/benchmarks/deepresearch_bench/data/drb_full_dataset.json
      structure:
        question_key: question
        answer_key: expected_output
        generated_answer_key: generated_answer
    #   filter:
    #     allowlist:
    #       field: # run sample of 16, for english only set 51-100, for all comment out the filter
    #         id: [88, 80, 84, 90, 59, 51, 94, 96, 91, 99, 93, 86, 67, 100, 72, 76]

  evaluators:
    race:
      _type: drb_race_evaluator
      llm_name: judge_llm
      criteria_file: frontends/benchmarks/deepresearch_bench/data/criteria.jsonl

In [ ]:
# If using OpenAI, you can set the API key in the environment variable OPENAI_API_KEY
# import getpass

# if "OPENAI_API_KEY" not in os.environ:
#     openai_api_key = getpass.getpass("Enter your OpenAI API key: ")
#     os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
!.venv/bin/nat eval --config_file frontends/benchmarks/deepresearch_bench/configs/config_deep_research_bench.yml

### Interpreting results

Results are written to the `output_dir` in the eval config (e.g. `frontends/benchmarks/deepresearch_bench/results/`).  

- **RACE** – Report quality (comprehensiveness, insight, instruction following, readability).  
- **FACT** – Citation accuracy and effective citations (requires Jina API for URL scraping if enabled).  

Check the config file for `eval.general.output_dir` and any `filter` settings that limit how many dataset items are run.

<a id="profiling"></a>
## Profiling and tracing

To **profile** or **trace** runs (e.g. Phoenix, Weave), enable the corresponding block in your workflow config under `general.telemetry` and set the required env vars (e.g. Phoenix endpoint, Weave project). Then run with `nat run` or `nat eval` as usual; traces will be sent to the configured backend.

Example snippet in a config (uncomment and set endpoint/project as needed):

```yaml
general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: dev
```

Start Phoenix (or your tracer) separately, then run the workflow; inspect traces in the Phoenix UI.

<a id="next"></a>
## Next steps
In the next notebook, **[2_Deep_Researcher_Customization](2_Deep_Researcher_Customization.ipynb)**, you will customize the deep researcher:

1. **Add a tool** — Install and configure the paper search tool (Google Scholar via Serper) and attach it to the deep research agent.
2. **Customize models** — Use a different LLM for the orchestrator (e.g. a frontier model) and Nemotron for the planner and researcher.
3. **Locate and customize prompts** — Find where orchestrator, planner, and researcher prompts live and what template variables they use.
4. **Add the knowledge layer** — Enable document retrieval so the agent can search over your ingested documents alongside web and paper search.